# Airline Digital Experience Exploratory Analysis

This notebook documents the first exploratory analysis pass for the airline digital experience data platform. It shows data understanding before dashboard design: available datasets, missing values, duplicate checks, delay distributions, route disruption, airport disruption, and passenger communication context.

Run the raw -> bronze -> silver -> gold pipeline before opening this notebook.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

SILVER_DIR = Path("../data/silver")
GOLD_DIR = Path("../data/gold")

## Load Available Datasets

Silver datasets are used for data quality checks. Gold datasets are used for business-facing metrics and dashboard validation.

In [ ]:
silver_tables = {
    "flights": pd.read_parquet(SILVER_DIR / "flights"),
    "airports": pd.read_parquet(SILVER_DIR / "airports"),
    "weather": pd.read_parquet(SILVER_DIR / "weather"),
    "passenger_events": pd.read_parquet(SILVER_DIR / "passenger_events"),
}

gold_tables = {
    "flight_performance": pd.read_parquet(GOLD_DIR / "gold_flight_performance"),
    "route_performance": pd.read_parquet(GOLD_DIR / "gold_route_performance"),
    "airport_disruption": pd.read_parquet(GOLD_DIR / "gold_airport_disruption"),
    "passenger_communication": pd.read_parquet(GOLD_DIR / "gold_passenger_communication"),
}

pd.DataFrame(
    [
        {"layer": "silver", "dataset": name, "rows": len(frame), "columns": len(frame.columns)}
        for name, frame in silver_tables.items()
    ]
    + [
        {"layer": "gold", "dataset": name, "rows": len(frame), "columns": len(frame.columns)}
        for name, frame in gold_tables.items()
    ]
)

## Missing Values And Duplicate Checks

The goal is to make data quality visible before dashboarding. Missing enrichment should not be hidden behind aggregates.

In [ ]:
quality_rows = []
for layer_name, tables in {"silver": silver_tables, "gold": gold_tables}.items():
    for dataset_name, frame in tables.items():
        quality_rows.append(
            {
                "layer": layer_name,
                "dataset": dataset_name,
                "rows": len(frame),
                "duplicate_rows": int(frame.duplicated().sum()),
                "missing_values": int(frame.isna().sum().sum()),
                "missing_value_rate": float(frame.isna().mean().mean()),
            }
        )

quality_summary = pd.DataFrame(quality_rows)
quality_summary

## Delay Distribution

NumPy is used here for simple distribution statistics that can inform dashboard KPI thresholds.

In [ ]:
flight_performance = gold_tables["flight_performance"]
delay_values = flight_performance["departure_delay_minutes"].to_numpy(dtype=float)

pd.DataFrame(
    {
        "metric": ["min", "p50", "p90", "max", "mean"],
        "departure_delay_minutes": [
            np.min(delay_values),
            np.percentile(delay_values, 50),
            np.percentile(delay_values, 90),
            np.max(delay_values),
            np.mean(delay_values),
        ],
    }
)

## Business Observations

1. Route-level delay rate is a better dashboard ranking metric than raw delayed-flight count for small demo data, because it normalizes by the number of flights.
2. Airport disruption should stay visible as a separate view because origin airports can explain operational bottlenecks that are hidden in route-only aggregates.
3. Passenger communication metrics connect operational disruption with Digital Hangar product surfaces, making the dashboard relevant beyond pure flight operations.
4. Missing enrichment should be tracked explicitly because public API data can be incomplete or unavailable in local/demo runs.

In [ ]:
gold_tables["route_performance"].sort_values(
    ["delay_rate", "average_departure_delay_minutes"], ascending=[False, False]
)[
    [
        "route",
        "total_flights",
        "delayed_flights",
        "cancelled_flights",
        "delay_rate",
        "average_departure_delay_minutes",
    ]
]

In [ ]:
gold_tables["airport_disruption"].sort_values(
    ["delay_rate", "cancelled_departures", "average_departure_delay_minutes"],
    ascending=[False, False, False],
)[
    [
        "origin_airport",
        "airport_name",
        "city",
        "country",
        "total_departures",
        "delayed_departures",
        "cancelled_departures",
        "delay_rate",
    ]
]

In [ ]:
gold_tables["passenger_communication"].sort_values(
    ["delayed_flight_events", "cancelled_flight_events", "event_count"],
    ascending=[False, False, False],
)[
    [
        "route",
        "event_type",
        "channel",
        "event_count",
        "delayed_flight_events",
        "cancelled_flight_events",
    ]
]

## Candidate Dashboard Metrics

- Total flights
- Delayed flights
- Cancelled flights
- Delay rate
- Average departure delay
- Top disrupted routes
- Airport disruption ranking
- Passenger communication events around disruptions